# Option A — Fast Path: Train 3 Backbones + Segmentation
**Self-contained. No dependency on the main notebook.**

Trains only Baseline / G1 / PNG backbones (skips G2–G5), saves checkpoints after each,
then immediately runs ADE20K segmentation.

- Images fed to ViT: **224×224** (model hard-requires this)
- Masks / logits upsampled to: **512×512**

## 0. Install Requirements

In [ ]:
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

# On Kaggle, torch with CUDA is pre-installed — skip torch reinstall.
# Only install missing packages.
pip('timm==1.0.25')
pip('kagglehub')
pip('tqdm', 'Pillow', 'numpy', 'matplotlib', 'scipy')

import torch
print(f'torch {torch.__version__} | CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: No CUDA detected — make sure the Kaggle GPU accelerator is enabled.')


In [ ]:
import os, types, random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from torch.utils.data import DataLoader, Dataset
import timm
from timm.layers import Attention

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

# ── Fetch data & checkpoints via kagglehub ────────────────────────────────────
# On Kaggle, credentials are built-in — no need to set KAGGLE_USERNAME/KEY.
import kagglehub

print('Downloading ADE20K ...')
_ade_root = Path(kagglehub.dataset_download('ipythonx/ade20k-scene-parsing'))
print(f'  ADE20K root: {_ade_root}')

print('Downloading backbone checkpoints ...')
_ckpt_root = Path(kagglehub.notebook_output_download('shashwatchaturvedi35/dl-project'))
print(f'  Checkpoint root: {_ckpt_root}')

# ── Resolve checkpoint path ───────────────────────────────────────────────────
_ckpt_candidates = list(_ckpt_root.rglob('checkpoints_B'))
CKPT_IN = _ckpt_candidates[0] if _ckpt_candidates else _ckpt_root
print(f'  Checkpoints : {CKPT_IN}')
print(f'  .pt files   : {[p.name for p in CKPT_IN.glob("*.pt")]}')

# ── Resolve ADE20K path ───────────────────────────────────────────────────────
_ade_candidates = list(_ade_root.rglob('ADEChallengeData2016'))
ADE_BASE = _ade_candidates[0] if _ade_candidates else _ade_root
print(f'  ADE20K base : {ADE_BASE}')

CKPT_OUT = Path('./checkpoints_seg')
CKPT_OUT.mkdir(exist_ok=True)

print(f'\ntorch {torch.__version__} | timm {timm.__version__} | device {DEVICE}')


## 1. Paths & constants

In [3]:
# IN1K_BASE is only defined when ImageNet was downloaded (retraining path).
# Default to None — the train_cls cell will skip training if checkpoints exist.
IN1K_BASE = globals().get('IN1K_BASE', None)
IN1K_TRAIN = IN1K_BASE / 'train' if IN1K_BASE is not None else None

ADE_TRAIN_IMG = ADE_BASE / 'images/training'
ADE_VAL_IMG   = ADE_BASE / 'images/validation'
ADE_TRAIN_ANN = ADE_BASE / 'annotations/training'
ADE_VAL_ANN   = ADE_BASE / 'annotations/validation'

SLICE       = 0.1
EPOCHS_CLS  = 2
EPOCHS_SEG  = 10
VIT_SIZE    = 224
MASK_SIZE   = 512
ADE_CLASSES = 150

for p in [ADE_TRAIN_IMG, ADE_VAL_IMG, ADE_VAL_ANN]:
    print(f'  {p.name:<40} {"OK" if p.exists() else "MISSING — check path"}')
if IN1K_TRAIN is not None:
    print(f'  {"IN1K_TRAIN":<40} {"OK" if IN1K_TRAIN.exists() else "MISSING"}')
else:
    print(f'  {"IN1K_TRAIN":<40} SKIPPED (checkpoints will be loaded)')


  training                                 OK
  validation                               OK
  validation                               OK
  IN1K_TRAIN                               SKIPPED (checkpoints will be loaded)


## 2. Gate code

In [4]:
class GateParams(nn.Module):
    def __init__(self, dim, num_heads, png=False):
        super().__init__()
        self.W = nn.Parameter(torch.empty(dim, num_heads).normal_(std=0.02))
        self.png = png
        if png:
            self.e    = nn.Parameter(torch.ones(num_heads))
            self.beta = nn.Parameter(torch.zeros(1))

    def gate(self, x):
        g = x @ self.W
        if self.png:
            g = g - self.beta * (x.norm(dim=-1, keepdim=True) + 1e-6) * self.e
        return torch.sigmoid(g)


def _patched_forward(mod, params, pos):
    def forward(self, x, attn_mask=None):
        B, N, C = x.shape
        H, D = self.num_heads, self.head_dim
        qkv = self.qkv(x).reshape(B, N, 3, H, D).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        q, k = self.q_norm(q), self.k_norm(k)
        if pos == 'G3': k = k * params.gate(x).permute(0,2,1).unsqueeze(-1)
        if pos == 'G4': q = q * params.gate(x).permute(0,2,1).unsqueeze(-1)
        if pos == 'G2': v = v * params.gate(x).permute(0,2,1).unsqueeze(-1)
        out = F.scaled_dot_product_attention(
            q, k, v, attn_mask=attn_mask,
            dropout_p=self.attn_drop.p if self.training else 0.)
        if pos in ('G1', 'PNG'):
            G = params.gate(x)
            out = (out.transpose(1,2) * G.unsqueeze(-1)).transpose(1,2)
            if getattr(self, '_capture', False):
                self._last_gate   = G.detach()
                self._last_x_norm = (x.norm(dim=-1, keepdim=True) + 1e-6).detach()
        x = out.transpose(1,2).reshape(B, N, C)
        if pos == 'G5':
            x = (x.view(B,N,H,D) * params.gate(x).unsqueeze(-1)).view(B,N,C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x
    return types.MethodType(forward, mod)


def inject_gates(model, pos='G1', png=False):
    gate_list = []
    p_label = 'PNG' if png else pos
    for _, mod in model.named_modules():
        if isinstance(mod, Attention):
            p = GateParams(mod.qkv.in_features, mod.num_heads, png=png)
            mod.forward = _patched_forward(mod, p, p_label)
            gate_list.append(p)
    model.gate_params = nn.ModuleList(gate_list)
    return model


def build_vit(pos='baseline', png=False, pretrained=True):
    """Build ViT. Pass pretrained=False when loading from a local checkpoint
    to avoid an unnecessary (and potentially failing) network download."""
    m = timm.create_model('vit_base_patch16_224', pretrained=pretrained, num_classes=1000)
    if pos != 'baseline':
        m = inject_gates(m, pos=pos, png=png)
    for p in m.parameters():          p.requires_grad_(False)
    for p in m.head.parameters():     p.requires_grad_(True)
    if hasattr(m, 'gate_params'):
        for p in m.gate_params.parameters(): p.requires_grad_(True)
    return m.to(DEVICE)

print('Gate code ready.')


Gate code ready.


## 3. ImageNet-1K dataloader (for backbone fine-tuning)

In [ ]:
train_tfm = T.Compose([
    T.RandomResizedCrop(224), T.RandomHorizontalFlip(),
    T.PILToTensor(),
    T.ConvertImageDtype(torch.float32),
    T.Normalize(MEAN, STD),
])
val_tfm = T.Compose([
    T.Resize(256), T.CenterCrop(224),
    T.PILToTensor(),
    T.ConvertImageDtype(torch.float32),
    T.Normalize(MEAN, STD),
])

class ImageSamples(Dataset):
    def __init__(self, samples, tfm):
        self.samples, self.tfm = samples, tfm
    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        path, label = self.samples[i]
        return self.tfm(Image.open(path).convert('RGB')), label

def build_imagenet_samples(root, frac):
    class_dirs   = sorted(p for p in Path(root).iterdir() if p.is_dir())
    class_to_idx = {d.name: i for i, d in enumerate(class_dirs)}
    samples = []
    for d in class_dirs:
        files = sorted(d.glob('*.JPEG')) + sorted(d.glob('*.jpg'))
        keep  = max(1, int(len(files) * frac))
        for f in files[:keep]:
            samples.append((str(f), class_to_idx[d.name]))
    return samples

if IN1K_TRAIN is not None and IN1K_TRAIN.exists():
    print('Scanning ImageNet-1K train ...')
    all_samples = build_imagenet_samples(IN1K_TRAIN, SLICE)
    random.shuffle(all_samples)
    split      = int(len(all_samples) * 0.9)
    train_samp = all_samples[:split]
    val_samp   = all_samples[split:]
    in1k_train_loader = DataLoader(ImageSamples(train_samp, train_tfm), 64,
                                   shuffle=True,  num_workers=2, pin_memory=True)
    in1k_val_loader   = DataLoader(ImageSamples(val_samp,   val_tfm),   64,
                                   shuffle=False, num_workers=2, pin_memory=True)
    print(f'ImageNet-1K  train {len(train_samp):,} | val {len(val_samp):,}  ({SLICE*100:.0f}% slice)')
else:
    in1k_train_loader = None
    in1k_val_loader   = None
    print('ImageNet-1K loaders SKIPPED — checkpoints will be loaded from disk.')


## 4. ADE20K dataloader
Images resized to **224×224** for ViT; masks kept at **512×512**.

In [6]:
def pil_to_tensor(img):
    """Convert PIL RGB image to float tensor without numpy."""
    import struct
    w, h = img.size
    buf = img.tobytes()                                      # raw bytes
    t = torch.frombuffer(bytearray(buf), dtype=torch.uint8) # bytearray avoids numpy
    t = t.reshape(h, w, 3).permute(2, 0, 1).float() / 255.0
    return t

def pil_mask_to_tensor(mask_pil):
    """Convert PIL palette/L mask to int64 tensor without numpy."""
    mask_pil = mask_pil.convert('I')                        # int32 mode
    buf = mask_pil.tobytes()
    t = torch.frombuffer(bytearray(buf), dtype=torch.int32)
    return t.reshape(mask_pil.size[1], mask_pil.size[0]).long()

MEAN_T = torch.tensor(MEAN).view(3, 1, 1)
STD_T  = torch.tensor(STD).view(3, 1, 1)

class ADE20KDataset(Dataset):
    def __init__(self, img_dir, ann_dir, frac=1.0):
        imgs = sorted(img_dir.glob('*.jpg'))
        anns = sorted(ann_dir.glob('*.png'))
        n = max(1, int(len(imgs) * frac))
        self.imgs, self.anns = imgs[:n], anns[:n]

    def __len__(self): return len(self.imgs)

    def __getitem__(self, i):
        img = Image.open(self.imgs[i]).convert('RGB').resize(
            (VIT_SIZE, VIT_SIZE), Image.BILINEAR)
        img = (pil_to_tensor(img) - MEAN_T) / STD_T

        mask = Image.open(self.anns[i]).resize(
            (MASK_SIZE, MASK_SIZE), Image.NEAREST)
        mask = pil_mask_to_tensor(mask) - 1   # 1-150 → 0-149, 0 → -1 (ignore)
        return img, mask

ade_train = ADE20KDataset(ADE_TRAIN_IMG, ADE_TRAIN_ANN, frac=SLICE)
ade_val   = ADE20KDataset(ADE_VAL_IMG,   ADE_VAL_ANN,   frac=SLICE)
ade_train_loader = DataLoader(ade_train, 16, shuffle=True,  num_workers=0, pin_memory=True)
ade_val_loader   = DataLoader(ade_val,   16, shuffle=False, num_workers=0, pin_memory=True)
print(f'ADE20K  train {len(ade_train):,} | val {len(ade_val):,}  ({SLICE*100:.0f}% slice)')
print(f'ViT input: {VIT_SIZE}×{VIT_SIZE}  |  Mask size: {MASK_SIZE}×{MASK_SIZE}')

# Quick sanity check
_img, _mask = ade_train[0]
print(f'img shape={tuple(_img.shape)} dtype={_img.dtype}  '
      f'mask shape={tuple(_mask.shape)} dtype={_mask.dtype}  '
      f'mask range=[{_mask.min()},{_mask.max()}]')


ADE20K  train 2,021 | val 200  (10% slice)
ViT input: 224×224  |  Mask size: 512×512
img shape=(3, 224, 224) dtype=torch.float32  mask shape=(512, 512) dtype=torch.int64  mask range=[-1,149]


## 5. Train 3 backbones — with checkpoint saving after each
If a checkpoint already exists it is loaded instead of re-training.

In [7]:
@torch.no_grad()
def evaluate_cls(model, loader, num_classes=1000):
    """Returns acc, macro-F1, macro-precision, macro-recall."""
    model.eval()
    all_preds, all_labels = [], []
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        all_preds.append(model(x).argmax(1).cpu())
        all_labels.append(y.cpu())
    preds  = torch.cat(all_preds)
    labels = torch.cat(all_labels)

    acc = (preds == labels).float().mean().item()

    tp = torch.zeros(num_classes)
    fp = torch.zeros(num_classes)
    fn = torch.zeros(num_classes)
    for c in range(num_classes):
        pred_c = preds  == c
        true_c = labels == c
        tp[c]  = (pred_c & true_c).sum()
        fp[c]  = (pred_c & ~true_c).sum()
        fn[c]  = (~pred_c & true_c).sum()

    present   = (tp + fn) > 0
    prec_c    = tp / (tp + fp + 1e-8)
    rec_c     = tp / (tp + fn + 1e-8)
    f1_c      = 2 * prec_c * rec_c / (prec_c + rec_c + 1e-8)
    precision = prec_c[present].mean().item()
    recall    = rec_c[present].mean().item()
    f1        = f1_c[present].mean().item()
    return acc, f1, precision, recall


def train_cls(model, epochs, name, ckpt_path=None):
    if in1k_train_loader is None:
        raise RuntimeError('ImageNet loaders not available — cannot train. '
                           'Make sure IN1K_BASE is set and the dataset is downloaded.')
    crit  = nn.CrossEntropyLoss()
    opt   = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad], lr=1e-3, weight_decay=0.05)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    log   = {'loss': [], 'acc': [], 'f1': [], 'precision': [], 'recall': []}
    for ep in range(1, epochs+1):
        model.train()
        for n, mod in model.named_modules():
            if 'gate_params' not in n and 'head' not in n and isinstance(mod, nn.LayerNorm):
                mod.eval()
        total_loss = 0
        for x, y in tqdm(in1k_train_loader, desc=f'{name} ep{ep}', leave=False):
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            loss = nn.CrossEntropyLoss()(model(x), y)
            loss.backward(); opt.step()
            total_loss += loss.item()
        acc, f1, prec, rec = evaluate_cls(model, in1k_val_loader)
        sched.step()
        log['loss'].append(total_loss / len(in1k_train_loader))
        log['acc'].append(acc); log['f1'].append(f1)
        log['precision'].append(prec); log['recall'].append(rec)
        print(f'  {name} ep{ep}  loss={log["loss"][-1]:.4f}  acc={acc:.4f}  '
              f'f1={f1:.4f}  prec={prec:.4f}  rec={rec:.4f}')
        if ckpt_path is not None:
            torch.save(model.state_dict(), ckpt_path)
    return log


# (pos, png, checkpoint filename in CKPT_IN, display name)
backbone_configs = [
    ('baseline', False, 'vit_baseline.pt', 'Baseline'),
    ('G1',       False, 'vit_g1.pt',       'G1 Gate'),
    ('G1',       True,  'vit_png.pt',       'PNG Gate'),
]

cls_results = {}
for pos, png, ckpt_name, name in backbone_configs:
    ckpt_path = CKPT_IN / ckpt_name
    print(f'\n── {name} ──')
    if ckpt_path.exists():
        # Load from read-only input — no network needed
        print(f'  Loading backbone: {ckpt_path}')
        model = build_vit(pos, png, pretrained=False)
        model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
        # Skip classification eval (no ImageNet val loader needed here)
        log = {'loss': [], 'acc': [0.0], 'f1': [0.0], 'precision': [0.0], 'recall': [0.0]}
        if in1k_val_loader is not None:
            acc, f1, prec, rec = evaluate_cls(model, in1k_val_loader)
            log = {'loss': [], 'acc': [acc], 'f1': [f1], 'precision': [prec], 'recall': [rec]}
            print(f'  Loaded — acc={acc:.4f}  f1={f1:.4f}  prec={prec:.4f}  rec={rec:.4f}')
        else:
            print(f'  Loaded — (skipping cls eval, no ImageNet val loader)')
    else:
        # Fallback: train from scratch (requires ImageNet download)
        print(f'  WARNING: {ckpt_path} not found — training from pretrained weights')
        model = build_vit(pos, png, pretrained=True)
        log = train_cls(model, EPOCHS_CLS, name, ckpt_path=CKPT_OUT / ckpt_name)
    cls_results[name] = {'model': model, 'log': log}



── Baseline ──
  Loading backbone: /root/.cache/kagglehub/notebooks/shashwatchaturvedi35/dl-project/output/versions/2/checkpoints_B/vit_baseline.pt


/tmp/ipykernel_3462/396834160.py:83: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))


  Loaded — (skipping cls eval, no ImageNet val loader)

── G1 Gate ──
  Loading backbone: /root/.cache/kagglehub/notebooks/shashwatchaturvedi35/dl-project/output/versions/2/checkpoints_B/vit_g1.pt
  Loaded — (skipping cls eval, no ImageNet val loader)

── PNG Gate ──
  Loading backbone: /root/.cache/kagglehub/notebooks/shashwatchaturvedi35/dl-project/output/versions/2/checkpoints_B/vit_png.pt
  Loaded — (skipping cls eval, no ImageNet val loader)


## 6. Segmentation head & helpers

In [ ]:
PATCH_GRID = VIT_SIZE // 16   # 14

class LinearSegHead(nn.Module):
    def __init__(self, embed_dim=768, num_classes=150):
        super().__init__()
        self.head = nn.Conv2d(embed_dim * 4, num_classes, kernel_size=1)

    def forward(self, feats, out_size):
        # feats: list of 4 tensors, each (B, 196, C=768)
        # Reshape each to (B, C, 14, 14), concatenate, apply head conv,
        # then upsample the 150-channel logits — avoids INT_MAX overflow.
        maps = []
        for f in feats:
            B, N, C = f.shape   # N = 196 = PATCH_GRID²
            f = f.transpose(1, 2).reshape(B, C, PATCH_GRID, PATCH_GRID)  # (B, C, 14, 14)
            maps.append(f)
        x = self.head(torch.cat(maps, dim=1))   # (B, num_classes, 14, 14)
        return F.interpolate(x, size=out_size, mode='bilinear', align_corners=False)


class SegModel(nn.Module):
    def __init__(self, vit, seg_head):
        super().__init__()
        self.vit      = vit
        self.seg_head = seg_head

    def forward(self, x):
        # x: (B, 3, 224, 224)
        feats = self.vit.get_intermediate_layers(x, n=[8, 9, 10, 11], reshape=False)
        # each feat: (B, 196, 768) — prefix tokens already stripped by timm
        return self.seg_head(feats, (MASK_SIZE, MASK_SIZE))


@torch.no_grad()
def mean_iou(model, loader):
    model.eval()
    inter = torch.zeros(ADE_CLASSES)
    union = torch.zeros(ADE_CLASSES)
    for imgs, masks in loader:
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        preds = model(imgs).argmax(1)
        valid = masks >= 0
        for c in range(ADE_CLASSES):
            pred_c = (preds == c) & valid
            true_c = (masks == c) & valid
            inter[c] += (pred_c & true_c).sum().cpu()
            union[c] += (pred_c | true_c).sum().cpu()
    iou = inter / (union + 1e-6)
    return iou[union > 0].mean().item()


def train_seg(vit_model, name, epochs=EPOCHS_SEG, lr=1e-4):
    seg_head = LinearSegHead(embed_dim=768, num_classes=ADE_CLASSES).to(DEVICE)
    model    = SegModel(vit_model, seg_head).to(DEVICE)

    for p in model.vit.parameters(): p.requires_grad_(False)
    if hasattr(model.vit, 'gate_params'):
        for p in model.vit.gate_params.parameters(): p.requires_grad_(True)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  trainable {trainable:,}')

    opt   = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=0.01)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    crit  = nn.CrossEntropyLoss(ignore_index=-1)

    log = {'loss': [], 'miou': []}
    for ep in range(1, epochs+1):
        model.train(); model.vit.eval()
        total_loss = 0
        for imgs, masks in tqdm(ade_train_loader, desc=f'seg {name} ep{ep}', leave=False):
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(imgs), masks)
            loss.backward(); opt.step()
            total_loss += loss.item()
        miou = mean_iou(model, ade_val_loader)
        sched.step()
        log['loss'].append(total_loss / len(ade_train_loader))
        log['miou'].append(miou)
        print(f'  seg {name} ep{ep}  loss={log["loss"][-1]:.4f}  mIoU={miou:.4f}')
    return model, log

print(f'Seg head ready. Patch grid {PATCH_GRID}×{PATCH_GRID} → upsample to {MASK_SIZE}×{MASK_SIZE}')
print(f'get_intermediate_layers output: (B, {PATCH_GRID**2}, 768) — patch tokens only')


## 7. Run segmentation

In [9]:
seg_results = {}
for name in ['Baseline', 'G1 Gate', 'PNG Gate']:
    print(f'\n── Seg: {name} ──')
    vit = cls_results[name]['model']
    seg_model, log = train_seg(vit, name, epochs=EPOCHS_SEG)
    seg_results[name] = {'model': seg_model, 'log': log}


── Seg: Baseline ──
  trainable 460,950


seg Baseline ep1:   0%|          | 0/127 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 8. Fig C — mIoU comparison

In [ ]:
seg_names = list(seg_results.keys())
mious     = [max(seg_results[n]['log']['miou']) * 100 for n in seg_names]
colors_s  = ['#888', '#4C72B0', '#DD8452']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
bars = ax1.bar(seg_names, mious, color=colors_s, edgecolor='k', linewidth=0.6)
for bar, m in zip(bars, mious):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             f'{m:.2f}', ha='center', va='bottom', fontsize=9)
ax1.set_ylabel('mIoU (%) — ADE20K val')
ax1.set_title('Segmentation: mIoU Comparison', fontweight='bold')
ax1.set_ylim(min(mious)*0.95, max(mious)*1.04)
ax1.grid(axis='y', alpha=0.3)

for n, c in zip(seg_names, colors_s):
    ax2.plot(range(1, EPOCHS_SEG+1), [v*100 for v in seg_results[n]['log']['miou']],
             'o-', color=c, label=n)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('mIoU (%)')
ax2.set_title('Segmentation Training Curves', fontweight='bold')
ax2.legend(); ax2.grid(alpha=0.3)

fig.suptitle(f'ADE20K Segmentation (Linear Probe, ViT@{VIT_SIZE}→mask@{MASK_SIZE})',
             fontsize=12, fontweight='bold')
fig.tight_layout()
fig.savefig('figC_seg_miou.pdf', bbox_inches='tight')
plt.show()
print('Saved figC_seg_miou.pdf')

## 9. Fig G — Qualitative examples

In [ ]:
rng_c = torch.Generator().manual_seed(0)
CMAP  = torch.cat([
    torch.zeros(1, 3, dtype=torch.uint8),
    torch.randint(50, 255, (150, 3), dtype=torch.uint8, generator=rng_c)
], dim=0)  # (151, 3)

def seg_to_rgb(mask_t):
    """mask_t: (H, W) int64 tensor, values -1..149"""
    idx = (mask_t + 1).clamp(0, 150)           # -1→0 (black), 0-149→1-150
    return CMAP[idx].numpy()                    # (H, W, 3) uint8 — numpy only for plt

def denorm_img(t):
    m = torch.tensor(MEAN).view(3,1,1)
    s = torch.tensor(STD).view(3,1,1)
    return (t.cpu()*s + m).clamp(0,1).permute(1,2,0).numpy()

val_imgs, val_masks = next(iter(DataLoader(ade_val, batch_size=4, shuffle=False, num_workers=0)))

def get_pred(seg_model, imgs):
    seg_model.eval()
    with torch.no_grad():
        return seg_model(imgs.to(DEVICE)).argmax(1).cpu()

pred_b = get_pred(seg_results['Baseline']['model'], val_imgs)
pred_g = get_pred(seg_results['G1 Gate']['model'],  val_imgs)
pred_p = get_pred(seg_results['PNG Gate']['model'],  val_imgs)

fig, axes = plt.subplots(4, 5, figsize=(15, 12))
for j, t in enumerate(['Image', 'GT Mask', 'Baseline', 'G1 Gate', 'PNG Gate']):
    axes[0, j].set_title(t, fontsize=10, fontweight='bold')
for i in range(4):
    axes[i, 0].imshow(denorm_img(val_imgs[i]));            axes[i, 0].axis('off')
    axes[i, 1].imshow(seg_to_rgb(val_masks[i]));           axes[i, 1].axis('off')
    axes[i, 2].imshow(seg_to_rgb(pred_b[i]));              axes[i, 2].axis('off')
    axes[i, 3].imshow(seg_to_rgb(pred_g[i]));              axes[i, 3].axis('off')
    axes[i, 4].imshow(seg_to_rgb(pred_p[i]));              axes[i, 4].axis('off')

fig.suptitle(f'ADE20K Segmentation Predictions (ViT@{VIT_SIZE}→mask@{MASK_SIZE})',
             fontsize=12, fontweight='bold')
fig.tight_layout()
fig.savefig('figG_seg_qual.pdf', bbox_inches='tight')
plt.show()
print('Saved figG_seg_qual.pdf')


## 10. Summary

In [ ]:
print('=' * 70)
print('  OPTION A RESULTS')
print('=' * 70)

print(f'\n  {"Model":<20} {"Acc":>8} {"Precision":>10} {"Recall":>8} {"F1":>8}  (ImageNet-1K)')
print('  ' + '-'*57)
for name in cls_results:
    log  = cls_results[name]['log']
    acc  = max(log['acc'])
    prec = max(log['precision'])
    rec  = max(log['recall'])
    f1   = max(log['f1'])
    print(f'  {name:<20} {acc*100:>7.2f}% {prec*100:>9.2f}% {rec*100:>7.2f}% {f1*100:>7.2f}%')

print(f'\n  {"Model":<20} {"Best mIoU":>10}  (ADE20K)')
print('  ' + '-'*35)
for name in seg_results:
    miou = max(seg_results[name]['log']['miou'])
    print(f'  {name:<20} {miou*100:>9.2f}%')

print(f'\n  Checkpoints saved in: {CKPT_OUT.resolve()}')
print('=' * 70)


In [ ]:
# Save seg checkpoints for output
import shutil

for name, fname in [('Baseline', 'vit_seg_baseline.pt'),
                    ('G1 Gate',  'vit_seg_g1.pt'),
                    ('PNG Gate', 'vit_seg_png.pt')]:
    if name in seg_results:
        path = CKPT_OUT / fname
        torch.save(seg_results[name]['model'].seg_head.state_dict(), path)
        print(f'Saved {path}')

print('Done.')
